# 04 — Modelos: Small CNN na imagem e baselines tabulares

Notebook equivalente a `src/models/`. **Branch `omicsfootprint` (R10):** os treinos da main (os 4 modelos, a otimização,
o R1, as logísticas da PARTE C e os blocos B, C e D) saíram desta branch. Os resultados deles continuam em `results/` e
são só lidos, pelo `05_eval`, na comparação final. Aqui ficam a padronização, a Small CNN, a logística, a rede base dos
testes R (`profundo + adamw_dro`) e o treino do R10.

Decisões do usuário (2026-10-09): mutação em `log(1 + contagem)` antes de padronizar (feita no `03_images`
e registrada no `padronizacao.json`); rede em **Keras** (motor TensorFlow-CPU); a rede é a **Small CNN** do
projeto irmão.

Entrada: `data/processed/ofp/`, gerado pelo `03_images` desta branch (folds, tensor e padronização dos 16.993 genes de
1–22, tabela gene → ângulo e imagens do R10). Saída: só em `results/ofp/` (previsões fora do fold em `results/ofp/r10/` e
o `04_models.log`, com fold, época e memória).

In [1]:
import gc
import json
import os
import time
from pathlib import Path

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # silencia o log C++ do TF (avisos de CPU e de versão do tf.data); erro Python continua aparecendo
import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from scipy.special import softmax
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedShuffleSplit

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
# branch omicsfootprint: grava só em ofp/ (data/processed/ofp/ e results/ofp/); pode ler de fora
PROCESSED, RESULTS = RAIZ / "data" / "processed" / "ofp", RAIZ / "results" / "ofp"
RESULTS.mkdir(parents=True, exist_ok=True)  # o log() abaixo já grava aqui
SEED, K = 42, 5
CLASSES = ["CIN", "MSI", "GS", "EBV"]  # = classe_idx 0, 1, 2, 3
CANAIS = ["mutacao", "cnv", "metilacao"]
PROBS = [f"p_{c}" for c in CLASSES]
tf.config.experimental.enable_op_determinism()  # mesma semente, mesmo resultado

# o nbconvert só grava o notebook no fim: o progresso vai para um arquivo (`tail -f results/ofp/04_models.log`),
# com a memória do processo, porque a máquina tem 4,8 GB e a 1ª execução morreu por falta de memória
LOG = RESULTS / "04_models.log"


def log(msg, tela=True):
    mem = {l.split(":")[0]: int(l.split()[1]) // 1024 for l in open("/proc/self/status") if l.startswith(("VmRSS", "VmHWM"))}
    linha = f"{time.strftime('%H:%M:%S')} [RAM {mem['VmRSS']} MB, pico {mem['VmHWM']} MB] {msg}"
    with open(LOG, "a") as f:
        f.write(linha + "\n")
    if tela:
        print(linha)


log("=== nova execução ===", tela=False)


coorte = pd.read_csv(PROCESSED / "coorte_cv.csv")
tensor = np.load(PROCESSED / "tensor_bruto.npy")
padronizacao = {int(k): v for k, v in json.loads((PROCESSED / "padronizacao.json").read_text()).items()}
angulos = pd.read_csv(PROCESSED / "angulos_r10.csv")  # gene → ângulo do R10 (03_images): os 16.993 genes de 1–22
y, fold = coorte.classe_idx.to_numpy(), coorte.fold.to_numpy()
assert tensor.shape == (len(coorte), 3, len(angulos)) and set(padronizacao) == set(range(K))
assert all(s["mutacao"].get("transformacao") == "log1p" for s in padronizacao.values())  # json do 03_images atual
print(f"{len(coorte)} pacientes {dict(zip(CLASSES, np.bincount(y)))} · tensor {tensor.shape} · keras {keras.__version__}")

336 pacientes {'CIN': np.int64(202), 'MSI': np.int64(59), 'GS': np.int64(46), 'EBV': np.int64(29)} · tensor (336, 3, 16993) · keras 3.15.1


## Padronização — as mesmas funções do `03_images`

Copiadas, porque o notebook é autocontido. A conferência garante que a cópia bate com o `padronizacao.json`:
no treino de cada fold, todo canal fica com média ≈ 0. Se o `log1p` faltasse, a mutação sairia deslocada. Na branch, a padronização serve à
logística sem o X; a rede do R10 recebe a imagem em [0, 1], sem padronização.

In [2]:
def transformar(x):
    x = x.copy()
    x[:, 0] = np.log1p(x[:, 0])  # mutação em log(1 + contagem), como no 03_images
    return x


def padronizar(x, stats):
    x = transformar(x)
    media = np.array([stats[c]["media"] for c in CANAIS], np.float32)[None, :, None]
    desvio = np.array([stats[c]["desvio"] for c in CANAIS], np.float32)[None, :, None]
    return np.nan_to_num((x - media) / desvio, nan=0.0)


for k in range(K):
    assert np.allclose(padronizar(tensor[fold != k], padronizacao[k]).mean(axis=(0, 2)), 0, atol=1e-3), k

## Small CNN em Keras

A Small CNN do projeto irmão (`proj_mutacoes_classificacao_TCGA-STAD/src/models/cnn_classifier.py`, em torch),
reescrita em Keras. São 4 blocos *convolução 3 → BatchNorm → ReLU → MaxPool 2*, com 16 → 32 → 64 → 64 filtros,
seguidos de média global, Dropout 0,3 e uma camada densa de 4 saídas. A BatchNorm usa `momentum=0.9`, o
equivalente ao padrão do torch. A mesma função monta a versão 2D
(imagem) e a 1D (vetor). Na 1D, o filtro tem 3 posições em vez de 3×3, por isso ela tem menos parâmetros.

O Keras usa os canais **por último**: a imagem do R10 entra como `(256, 256, 3)`.

Treino (`treinar_rede_base()`, na seção da rede base; na main, a `treinar_cnn()` usa os hiperparâmetros do irmão):
- **Validação interna:** 15% do treino, estratificada, usada só para decidir quando parar. O fold de
  teste nunca é visto;
- **Parada antecipada:** pelo F1 macro da validação interna, paciência de 15 épocas, até 60 épocas. Os
  pesos voltam para os da melhor época;
- **Otimização:** a da rede base (AdamW e group DRO, descritos abaixo), com lr 1e-3 e lote de 32;
- **Sem aumento de dados:** girar ou espelhar a imagem tiraria cada gene do seu pixel.

In [3]:
def small_cnn(entrada, n_classes=4, dropout=0.3, filtros=(16, 32, 64, 64)):
    conv, pool, gap = ((layers.Conv2D, layers.MaxPooling2D, layers.GlobalAveragePooling2D) if len(entrada) == 3
                       else (layers.Conv1D, layers.MaxPooling1D, layers.GlobalAveragePooling1D))
    x = entrada_ = keras.Input(entrada)
    for f in filtros:
        x = conv(f, 3, padding="same")(x)
        x = layers.BatchNormalization(momentum=0.9)(x)  # = momentum 0,1 do torch; o 0,99 padrão do Keras demora a assentar
        x = layers.ReLU()(x)
        x = pool(2)(x)
    x = layers.Dropout(dropout)(gap()(x))
    return keras.Model(entrada_, layers.Dense(n_classes)(x))  # logits; a perda aplica o softmax


class F1MacroVal(keras.callbacks.Callback):
    # põe o F1 macro da validação interna nos logs; vem antes do EarlyStopping na lista, que lê o mesmo dicionário
    def __init__(self, x, y):
        super().__init__()
        self.x, self.y, self.f1s = x, y, []

    def on_epoch_end(self, epoch, logs=None):
        self.f1s.append(f1_score(self.y, self.model.predict(self.x, verbose=0).argmax(axis=1), average="macro"))
        logs["val_f1_macro"] = self.f1s[-1]
        log(f"    época {epoch + 1}: perda {logs['loss']:.3f} · F1 macro val {self.f1s[-1]:.3f}", tela=False)


n_param = small_cnn((256, 256, 3)).count_params()
assert n_param == 61476  # o mesmo da rede base na Hilbert: as convoluções e a Dense(4) não dependem do tamanho da imagem
print(f"Small CNN na imagem do R10 (256 × 256 × 3): {n_param:,} parâmetros")

Small CNN na imagem do R10 (256 × 256 × 3): 61,476 parâmetros


## Baselines tabulares

Mesma entrada da Small CNN, achatada: os 3 canais × 16.993 genes (na branch, sem o X) viram 50.979 colunas, sem formato de
imagem. Configurações iguais às do irmão (`src/eval/baselines.py`):
- **logística:** L2, C = 1, peso balanceado por classe.

O XGBoost fica só na main: o R10 usa a logística (sem o X).

In [4]:
def treinar_logistica(x_tr, y_tr, x_te, seed):
    clf = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000, random_state=seed)
    return clf.fit(x_tr, y_tr).predict_proba(x_te)

# Rede base dos testes R: `profundo + adamw_dro`

Decisão do usuário (2026-10-31): a Small CNN com AdamW e group DRO, treinada pela `treinar_rede_base()`.

- **AdamW** (relatório, seção 5.7): o Adam com o decaimento de peso separado do gradiente. No Keras, a cada passo cada peso perde
  lr × wd = 0,001 × 1,0 = 0,1% do valor. Em 480 passos (60 épocas de ~8 lotes), sem gradiente nenhum, sobrariam 62%. Bias e
  BatchNorm não decaem.
- **Group DRO** (SAGAWA et al., 2020; relatório, seção 5.4), com grupo = classe. A cada lote, o peso q de cada classe é
  multiplicado por exp(η × perda média da classe no lote) e renormalizado, e a perda do lote é Σ q × perda da classe. A classe
  que vai pior ganha peso aos poucos. Substitui o peso de classe fixo. Pelo relatório, o group DRO só funciona com regularização
  forte, por isso ele entra junto com o AdamW.

`rodar_o()` treina, por semente e fold, o que ainda não tem arquivo e grava as previsões fora do fold. Na branch, ela só aceita pasta dentro de `ofp/`.

As duas conferências no fim da célula: o group DRO num lote de 4 pacientes feito à mão, e 1 época da rede base em 40
pacientes, para que um erro de código apareça em segundos, e não depois de horas de treino.

In [5]:
SEMENTES = [SEED, 142, 242, 342, 442]  # as 5 sementes dos testes R


class GroupDRO(keras.losses.Loss):
    # group DRO online, grupo = classe: q_g ← q_g · exp(η · perda média da classe g no lote); perda = Σ_g q_g · perda_g
    def __init__(self, eta=0.01):
        super().__init__(reduction="sum")
        self.eta = eta
        self.q = tf.Variable(tf.fill([len(CLASSES)], 1 / len(CLASSES)), trainable=False)

    def call(self, y_true, logits):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        ce = tf.nn.sparse_softmax_cross_entropy_with_logits(y_true, logits)
        g = tf.one_hot(y_true, len(CLASSES))                # (lote, 4)
        n = tf.maximum(tf.reduce_sum(g, 0), 1.0)
        perda_g = tf.reduce_sum(g * ce[:, None], 0) / n     # classe ausente do lote: perda 0, q dela não sobe
        q = self.q.assign(self.q * tf.exp(self.eta * tf.stop_gradient(perda_g)))
        q = self.q.assign(q / tf.reduce_sum(q))
        return ce * tf.reduce_sum(g * q / n, 1)             # somado no lote = Σ_g q_g · perda_g


def treinar_rede_base(x_tr, y_tr, x_te, seed, epocas=60):
    # rede base dos testes R (decisão do usuário, 2026-10-31): Small CNN + AdamW + group DRO, sem peso de classe
    keras.backend.clear_session()  # sem isso, cada modelo novo acumula memória no TensorFlow
    gc.collect()
    tr, va = next(StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=seed).split(x_tr, y_tr))
    keras.utils.set_random_seed(seed)
    modelo = small_cnn(x_tr.shape[1:])
    otimizador = keras.optimizers.AdamW(1e-3, weight_decay=1.0)
    otimizador.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])  # só os kernels decaem
    modelo.compile(otimizador, GroupDRO())  # o DRO substitui o peso de classe
    f1_val = F1MacroVal(x_tr[va], y_tr[va])
    parada = keras.callbacks.EarlyStopping("val_f1_macro", mode="max", patience=15, restore_best_weights=True)
    modelo.fit(x_tr[tr], y_tr[tr], epochs=epocas, batch_size=32, callbacks=[f1_val, parada], verbose=0)
    log(f"  {len(f1_val.f1s)} épocas, melhor F1 macro na validação interna {max(f1_val.f1s):.3f}")
    return softmax(modelo.predict(x_te, verbose=0), axis=1)


def rodar_o(nome, semente, preparar, pasta=RESULTS, treinar=treinar_rede_base):
    # previsões fora do fold (336, 4) na ordem do coorte_cv.csv; reaproveita o que já existe em `pasta` e treina o resto
    # preparar(k): a entrada do fold k, já preparada com o treino do fold k; treinar: a rede base ou a treinar_logistica()
    assert "ofp" in pasta.parts, "branch omicsfootprint: grava só em ofp/"
    pasta.mkdir(parents=True, exist_ok=True)
    proba = np.empty((len(y), len(CLASSES)))
    for k in range(K):
        te, arq = fold == k, pasta / f"{nome}_s{semente}_fold{k}.csv"
        if not arq.exists():
            log(f"{nome} · semente {semente} · fold {k}")
            x = preparar(k)
            x_tr, x_te = x[~te], x[te]
            del x  # não guardar a entrada inteira do fold (264 MB no R10) durante o treino
            p = treinar(x_tr, y[~te], x_te, semente + k)
            pd.DataFrame(p, columns=PROBS).assign(amostra=coorte.amostra[te].to_numpy()).to_csv(arq, index=False)
            del x_tr, x_te
        proba[te] = pd.read_csv(arq).set_index("amostra").loc[coorte.amostra[te], PROBS].to_numpy()
    return proba


# autoteste do group DRO: num lote em que a classe 1 vai mal, o peso dela sobe, e a perda é Σ q_g · perda_g
dro = GroupDRO(eta=1.0)
yt, lg = tf.constant([0, 0, 1, 1]), tf.constant([[3.0, 0, 0, 0]] * 4)
total, q = float(dro(yt, lg)), dro.q.numpy()
ce = tf.nn.sparse_softmax_cross_entropy_with_logits(yt, lg).numpy()
assert q[1] > q[0] > q[2] == q[3] and np.isclose(q.sum(), 1)
assert np.isclose(total, q[0] * ce[:2].mean() + q[1] * ce[2:].mean(), rtol=1e-5)

# autoteste da rede base: 1 época em 40 pacientes de treino do fold 0, para pegar erro antes das horas de treino
sub = np.concatenate([np.flatnonzero((y == c) & (fold != 0))[:10] for c in range(4)])
x = np.load(PROCESSED / "r10_imagens" / "r10_fold0.npy")[sub].astype(np.float32) / 16  # imagem do R10 em [0, 1]
p = treinar_rede_base(x, y[sub], x[:3], SEED, epocas=1)
assert p.shape == (3, 4) and np.allclose(p.sum(axis=1), 1, atol=1e-4)
del x, p

12:20:20 [RAM 1158 MB, pico 1715 MB]   1 épocas, melhor F1 macro na validação interna 0.125


# R10 — treino (pré-registro de 2026-11-02)

Diário, "[R10] Pré-registro do R10, do R10-p e da logística sem o X". Previsões fora do fold em `results/ofp/r10/`, nesta
ordem:
- **logística sem o X:** a entrada atual da logística (z por canal, C = 1) só com os 16.993 genes: 50.979 valores;
- **R10:** a rede base na imagem circular de cada fold (contagem / 16, em [0, 1]), 5 sementes (semente + k no fold k);
- **R10-p:** a mesma rede e as mesmas sementes, na imagem com o sorteio gene → ângulo da semente.

A `rodar_o()` reaproveita o arquivo que já existe: se a entrada mudar, apague `results/ofp/r10/` antes. O progresso vai
para `results/ofp/04_models.log` (fold, época e memória). A avaliação (Perm(R10), diferenças com IC e BBC-par) fica no
`05_eval`; aqui, só a conferência do pior F1 e do CIN → GS por semente.

In [6]:
R10 = RESULTS / "r10"  # previsões fora do fold do R10, do R10-p e da logística sem o X
IMG = PROCESSED / "r10_imagens"


def imagens(nome):
    # preparar(k) da rodar_o(): o conjunto de imagens do fold k, da contagem 0–16 para [0, 1]
    def preparar(k):
        x = np.load(IMG / f"{nome}_fold{k}.npy").astype(np.float32)
        x /= 16  # no lugar, para não guardar duas cópias de 264 MB
        return x
    return preparar


log("=== R10: logística sem o X, R10 e R10-p ===")
t0 = time.time()
PO = {("logística sem o X", SEED): rodar_o("logistica_sem_x", SEED, lambda k: padronizar(tensor, padronizacao[k]).reshape(len(y), -1),
                                          pasta=R10, treinar=treinar_logistica)}
for s in SEMENTES:
    PO[("R10", s)] = rodar_o("R10", s, imagens("r10"), pasta=R10)
for s in SEMENTES:
    PO[("R10-p", s)] = rodar_o("R10-p", s, imagens(f"r10p_s{s}"), pasta=R10)
log(f"R10 terminado em {(time.time() - t0) / 60:.0f} min")

# conferência rápida, sem IC (a avaliação fica no 05_eval): pior F1 e CIN → GS por semente
CIN, GS = CLASSES.index("CIN"), CLASSES.index("GS")
conf = pd.DataFrame([{"teste": t, "semente": s,
                      "pior F1": f1_score(y, p.argmax(axis=1), average=None, labels=range(len(CLASSES))).min(),
                      "CIN → GS": int(((y == CIN) & (p.argmax(axis=1) == GS)).sum())} for (t, s), p in PO.items()])
conf.groupby("teste", sort=False)[["pior F1", "CIN → GS"]].agg(["mean", "std"]).round(3)

12:20:20 [RAM 1135 MB, pico 1715 MB] === R10: logística sem o X, R10 e R10-p ===


12:20:21 [RAM 1135 MB, pico 1715 MB] R10-p · semente 42 · fold 0


12:33:42 [RAM 2508 MB, pico 3491 MB]   44 épocas, melhor F1 macro na validação interna 0.814


12:33:44 [RAM 2273 MB, pico 3491 MB] R10-p · semente 42 · fold 1


12:47:41 [RAM 2588 MB, pico 3615 MB]   48 épocas, melhor F1 macro na validação interna 0.918


12:47:43 [RAM 2336 MB, pico 3615 MB] R10-p · semente 42 · fold 2


... (43 outputs left out)

17:34:24 [RAM 2525 MB, pico 3880 MB] R10-p · semente 442 · fold 4


17:49:38 [RAM 2819 MB, pico 3880 MB]   44 épocas, melhor F1 macro na validação interna 0.840


17:49:41 [RAM 2567 MB, pico 3880 MB] R10 terminado em 329 min


pior F1        CIN → GS       
                     mean    std     mean    std
teste                                           
logística sem o X   0.757    NaN     17.0    NaN
R10                 0.622  0.037     24.2  3.701
R10-p               0.599  0.019     28.2  5.167

# Resumo do log do treino do R10

Pedido do usuário (2026-11-02): uma linha por teste, semente e fold, com as épocas, o melhor F1 macro da validação interna, o tempo e a memória, juntando as duas execuções do `results/ofp/04_models.log` (1: 2026-11-02, logística sem o X e R10; 2: 2026-11-03, R10-p). O fold do R10-p interrompido em 2026-11-02 às 22:54 fica de fora. "RAM máx. no fold" é a maior memória lida nas linhas do fold; "pico do processo" é o pico acumulado da execução até o fim do fold.

In [7]:
import re


def resumo_log(linhas):
    # blocos do log: cabeçalho "teste · semente s · fold k", as épocas e, nas redes, "N épocas, melhor F1 …".
    # A logística não registra épocas: o bloco dela fecha no cabeçalho seguinte. Um bloco ainda aberto quando aparece
    # "=== nova execução ===" foi interrompido e fica de fora. Tempo = fechamento − cabeçalho (passa da meia-noite).
    tab, aberto, execucao = [], None, 0
    for l in linhas:
        m = re.match(r"(\d+):(\d+):(\d+) \[RAM (\d+) MB, pico (\d+) MB\] +(.*)", l)
        t, ram, pico, msg = int(m[1]) * 3600 + int(m[2]) * 60 + int(m[3]), int(m[4]), int(m[5]), m[6]
        cab = re.fullmatch(r"(\S+) · semente (\d+) · fold (\d)", msg)
        fim = re.fullmatch(r"(\d+) épocas, melhor F1 macro na validação interna ([\d.]+)", msg)
        if msg == "=== nova execução ===":
            aberto = None
        elif msg.startswith("=== R10"):
            execucao += 1
        if aberto and (fim or cab or msg.startswith("R10 terminado")):
            tab.append({**aberto, "t0": None, "épocas": int(fim[1]) if fim else None,
                        "melhor F1 val": float(fim[2]) if fim else None,
                        "min": round(((t - aberto["t0"]) % 86400) / 60, 2),
                        "RAM máx. no fold (MB)": max(aberto["ram"], ram if fim else 0), "pico do processo (MB)": pico})
            aberto = None
        elif aberto:
            aberto["ram"] = max(aberto["ram"], ram)
        if cab:
            aberto = {"execução": execucao, "teste": cab[1], "semente": int(cab[2]), "fold": int(cab[3]), "t0": t, "ram": ram}
    return pd.DataFrame(tab).drop(columns=["t0", "ram"]).drop_duplicates(["teste", "semente", "fold"], keep="last")


# autoteste: logística fecha no cabeçalho seguinte; rede que passa da meia-noite; fold interrompido e autoteste fora
t = resumo_log(["23:50:00 [RAM 100 MB, pico 100 MB] === R10: teste ===",
                "23:50:00 [RAM 100 MB, pico 100 MB] logistica_sem_x · semente 42 · fold 0",
                "23:50:09 [RAM 150 MB, pico 150 MB] R10 · semente 42 · fold 0",
                "23:59:59 [RAM 300 MB, pico 320 MB]     época 1: perda 1.000 · F1 macro val 0.500",
                "00:01:09 [RAM 250 MB, pico 320 MB]   2 épocas, melhor F1 macro na validação interna 0.500",
                "00:01:10 [RAM 200 MB, pico 320 MB] R10-p · semente 42 · fold 0",
                "00:01:20 [RAM 100 MB, pico 100 MB] === nova execução ===",
                "00:01:30 [RAM 150 MB, pico 150 MB]   1 épocas, melhor F1 macro na validação interna 0.100"])
assert t.teste.tolist() == ["logistica_sem_x", "R10"] and t["min"].tolist() == [0.15, 11.0]
assert t["RAM máx. no fold (MB)"].tolist() == [100, 300] and t["épocas"].iloc[1] == 2

tab = resumo_log(LOG.read_text().splitlines())
assert tab.groupby("teste").size().to_dict() == {"logistica_sem_x": 5, "R10": 25, "R10-p": 25}  # nenhum fold faltando
print("Por teste (execução 1: 2026-11-02; execução 2: 2026-11-03):")
display(tab.groupby(["execução", "teste"], sort=False).agg(
    folds=("fold", "size"), épocas_média=("épocas", "mean"), épocas_mín=("épocas", "min"), épocas_máx=("épocas", "max"),
    F1_val_médio=("melhor F1 val", "mean"), horas=("min", lambda m: round(m.sum() / 60, 2)),
    RAM_máx_MB=("RAM máx. no fold (MB)", "max")).round(3))
with pd.option_context("display.max_rows", 100):
    display(tab.set_index(["execução", "teste", "semente", "fold"]))

Por teste (execução 1: 2026-11-02; execução 2: 2026-11-03):


folds  épocas_média  épocas_mín  épocas_máx  \
execução teste                                                          
1        logistica_sem_x      5           NaN         NaN         NaN   
         R10                 25         39.64        22.0        56.0   
2        R10-p               25         40.60        24.0        60.0   

                          F1_val_médio  horas  RAM_máx_MB  
execução teste                                             
1        logistica_sem_x           NaN   0.01        1161  
         R10                     0.901   5.25        3124  
2        R10-p                   0.901   5.47        3195

épocas  melhor F1 val    min  \
execução teste           semente fold                                 
1        logistica_sem_x 42      0        NaN            NaN   0.12   
                                 1        NaN            NaN   0.15   
                                 2        NaN            NaN   0.10   
                                 3        NaN            NaN   0.15   
                                 4        NaN            NaN   0.15   
         R10             42      0       56.0          0.836  15.47   
                                 1       38.0          0.918  11.08   
                                 2       39.0          0.909  11.62   
                                 3       50.0          0.934  16.15   
                                 4       29.0          0.903   9.97   
                         142     0       38.0          0.948  12.27   
                                 1       37.0          0.838  12.15   
                                 2       38.0          0.825  11.92   
                                 3       40.0          0.954  12.88   
                                 4       22.0          0.917   6.83   
                         242     0       31.0          0.874   8.80   
                                 1       53.0          0.865  14.78   
                                 2       28.0          0.908   9.00   
                                 3       26.0          0.927   7.95   
                                 4       38.0          0.865  12.07   
                         342     0       34.0          0.897   9.82   
                                 1       45.0          0.940  13.35   
                                 2       53.0          0.890  17.47   
                                 3       25.0          0.912   8.00   
                                 4       44.0          0.927  14.82   
                         442     0       39.0          0.909  12.40   
                                 1       47.0          0.967  16.22   
                                 2       39.0          0.948  12.05   
                                 3       53.0          0.934  21.00   
                                 4       49.0          0.775  16.65   
2        R10-p           42      0       44.0          0.814  13.35   
                                 1       48.0          0.918  13.95   
                                 2       29.0          0.909  10.17   
                                 3       29.0          0.890  11.28   
                                 4       60.0          0.954  20.70   
                         142     0       28.0          0.972   8.57   
                                 1       43.0          0.845  12.38   
                                 2       52.0          0.845  14.62   
                                 3       46.0          0.913  13.88   
                                 4       47.0          0.948  13.30   
                         242     0       40.0          0.832  12.23   
                                 1       60.0          0.890  17.10   
                                 2       26.0          0.927   7.20   
                                 3       45.0          0.916  12.33   
                                 4       47.0          0.886  13.83   
                         342     0       31.0          0.899   8.97   
                                 1       24.0          0.907   6.68   
                                 2       36.0          0.916   9.77   
                                 3       47.0          0.954  13.58   
                                 4       35.0          0.948  15.42   
                         442     0       36.0          0.867  14.77   
                                 1       29.0          0.896  11.93   
                                 2       29.0          0.927  13.98   
                                 3       60.0          0.903  23.08   
                                 4       44.0          0.840  1